In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df = pd.read_csv("/content/Telco-Customer-Churn.csv")
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,8270-XXXXX,Male,0,Yes,Yes,36,Yes,No,No,No,...,No,No,No internet service,No internet service,Month-to-month,Yes,Mailed check,45.41,1633.45,No
1,1860-XXXXX,Female,0,No,Yes,24,No,No phone service,No,Yes,...,Yes,No,No,No,One year,Yes,Bank transfer (automatic),115.37,2764.4,No
2,6390-XXXXX,Female,0,Yes,No,6,Yes,Yes,Fiber optic,No,...,No internet service,No internet service,No,No internet service,One year,Yes,Mailed check,74.30,444.97,No
3,6191-XXXXX,Male,0,No,No,66,Yes,No phone service,Fiber optic,No internet service,...,Yes,Yes,No internet service,No,One year,Yes,Electronic check,53.15,3511.4,Yes
4,6734-XXXXX,Male,0,Yes,No,4,Yes,No phone service,Fiber optic,No internet service,...,No,Yes,Yes,No internet service,One year,No,Electronic check,76.59,307.93,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,7546-XXXXX,Male,0,No,Yes,54,Yes,Yes,No,No internet service,...,Yes,No internet service,Yes,No,Month-to-month,Yes,Mailed check,54.44,2944.56,No
196,2986-XXXXX,Female,1,Yes,Yes,24,Yes,Yes,DSL,Yes,...,No,Yes,Yes,No,Month-to-month,Yes,Electronic check,114.41,2746.03,No
197,9338-XXXXX,Male,0,No,Yes,25,Yes,No phone service,No,No internet service,...,Yes,No internet service,No,No internet service,Two years,Yes,Electronic check,68.54,1718.21,No
198,3911-XXXXX,Male,1,Yes,No,71,Yes,No,No,Yes,...,No,No,Yes,No,Month-to-month,No,Mailed check,61.74,4382.58,No


In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        200 non-null    object 
 1   gender            200 non-null    object 
 2   SeniorCitizen     200 non-null    int64  
 3   Partner           200 non-null    object 
 4   Dependents        200 non-null    object 
 5   tenure            200 non-null    int64  
 6   PhoneService      200 non-null    object 
 7   MultipleLines     200 non-null    object 
 8   InternetService   200 non-null    object 
 9   OnlineSecurity    200 non-null    object 
 10  OnlineBackup      200 non-null    object 
 11  DeviceProtection  200 non-null    object 
 12  TechSupport       200 non-null    object 
 13  StreamingTV       200 non-null    object 
 14  StreamingMovies   200 non-null    object 
 15  Contract          200 non-null    object 
 16  PaperlessBilling  200 non-null    object 
 1

In [4]:
df.shape

(200, 21)

In [5]:
print(df['Churn'].value_counts())

Churn
No     130
Yes     70
Name: count, dtype: int64


In [6]:
# Remove customerID (not useful for prediction)
df.drop('customerID', axis=1, inplace=True)

In [7]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())



In [8]:
# Encode categorical variables
for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])

In [9]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,1,0,1,1,36,1,0,2,0,0,0,0,1,1,0,1,3,45.41,1633.45,0
1,0,0,0,1,24,0,1,2,2,2,2,0,0,0,1,1,0,115.37,2764.40,0
2,0,0,1,0,6,1,2,1,0,0,1,1,0,1,1,1,3,74.30,444.97,0
3,1,0,0,0,66,1,1,1,1,0,2,2,1,0,1,1,2,53.15,3511.40,1
4,1,0,1,0,4,1,1,1,1,1,0,2,2,1,1,0,2,76.59,307.93,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,0,0,1,54,1,2,2,1,2,2,1,2,0,0,1,3,54.44,2944.56,0
196,0,1,1,1,24,1,2,0,2,0,0,2,2,0,0,1,2,114.41,2746.03,0
197,1,0,0,1,25,1,1,2,1,1,2,1,0,1,2,1,2,68.54,1718.21,0
198,1,1,1,0,71,1,0,2,2,1,0,0,2,0,0,0,3,61.74,4382.58,0


In [10]:
# Features and target
X = df.drop('Churn', axis=1)
y = df['Churn']


In [11]:
# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [12]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.3))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)


Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - accuracy: 0.5312 - loss: 0.7502 - val_accuracy: 0.5000 - val_loss: 0.7470
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5391 - loss: 0.7717 - val_accuracy: 0.4688 - val_loss: 0.7291
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4609 - loss: 0.7939 - val_accuracy: 0.4688 - val_loss: 0.7138
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5469 - loss: 0.7355 - val_accuracy: 0.5000 - val_loss: 0.7013
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5781 - loss: 0.6745 - val_accuracy: 0.5000 - val_loss: 0.6907
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5312 - loss: 0.7432 - val_accuracy: 0.5312 - val_loss: 0.6831
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5781 - loss: 0.6891 - val_accuracy: 0.5625 - val_loss: 0.6794
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6641 - loss: 0.6692 - val_accuracy: 0.5938 - val_loss: 0.6773


In [15]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

# Predictions
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5750 - loss: 0.7737 
Test Accuracy: 0.57
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
              precision    recall  f1-score   support

           0       0.57      0.95      0.71        22
           1       0.67      0.11      0.19        18

    accuracy                           0.57        40
   macro avg       0.62      0.53      0.45        40
weighted avg       0.61      0.57      0.48        40

[[21  1]
 [16  2]]


In [16]:
# Example new customer data (dummy values, same order as features)
new_customer = np.array([[
    0,   # gender (0=Female,1=Male)
    0,   # SeniorCitizen
    1,   # Partner
    0,   # Dependents
    1,   # tenure (e.g., 1-72 months)
    1,   # PhoneService
    0,   # MultipleLines
    1,   # InternetService
    0,   # OnlineSecurity
    1,   # OnlineBackup
    0,   # DeviceProtection
    1,   # TechSupport
    0,   # StreamingTV
    1,   # StreamingMovies
    1,   # Contract
    0,   # PaperlessBilling
    1,   # PaymentMethod
    50.0,# MonthlyCharges
    200.0# TotalCharges
]])
new_customer = scaler.transform(new_customer)  # Scale like training data
churn_prob = model.predict(new_customer)[0][0]

print(f"\nChurn Probability for New Customer: {churn_prob:.2f}")
print("Prediction:", "Churn" if churn_prob > 0.5 else "No Churn")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

Churn Probability for New Customer: 0.35
Prediction: No Churn


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
